<a href="https://colab.research.google.com/github/jakartaresearch/quora-question-pairs/blob/master/notebook/R1_data_cleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Covers

- Remove NaN value
- Remove punctuation
- Remove hyperlinks
- Remove HTML tag
- Remove non-ascii character
- Remove single character
- Lowercase all text

In [ ]:
# Mount Google Drive
from google.colab import drive # import drive from google colab

ROOT = "/content/drive/"     # default location for the drive
drive.mount(ROOT)           # we mount the google drive at /content/drive

import pandas as pd
import numpy as np
import re
import string
import unicodedata
import nltk

from tqdm import tqdm_notebook
tqdm_notebook().pandas()
from bs4 import BeautifulSoup
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
data_path = 'drive/My Drive/Colab Notebooks/quora-question-pairs/data/quora_duplicate_questions.tsv'
data = pd.read_csv(data_path, sep='\t')

In [ ]:
data.head()

## Remove NaN value

In [ ]:
print('Jumlah NaN di question1 :', data.question1.isnull().sum())
print('Jumlah NaN di question2 :', data.question2.isnull().sum())

In [ ]:
data[pd.isna(data.question1)]

In [ ]:
data[pd.isna(data.question2)]

In [ ]:
index_drop = [data[pd.isna(data.question1)].index[0],
              data[pd.isna(data.question2)].index[0],
              data[pd.isna(data.question2)].index[1]]

In [ ]:
# Drop data which question1 or question2 have NaN value
data = data.drop(index_drop).reset_index(drop=True)

## Find out hyperlink

In [ ]:
def find_hyperlink(string): 
    url = re.findall('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', string) 
    return url 

In [ ]:
# Test code 
string = 'My Profile: https://auth.geeksforgeeks.org / user / Chinmoy % 20Lenka / articles in the portal of http://www.geeksforgeeks.org/' 
print("Urls: ", find_hyperlink(string)) 

In [ ]:
# Check hyperlink in 'question1' and 'question2'
total_hyperlink = 0

for i in data['question1']:
  result = find_hyperlink(i)
  if len(result) != 0 :
    total_hyperlink += 1
    print('Get Hyperlink :', result)

for i in data['question2']:
  result = find_hyperlink(i)
  if len(result) != 0 :
    total_hyperlink += 1
    print('Get Hyperlink :', result)

print('Total Hyperlink :', total_hyperlink)

# remove punctuation, remove html tag, remove non-ascii character, remove stopwords

In [ ]:
def remove_punctuation(text):
    import string
    table = str.maketrans(string.punctuation, ' '*len(string.punctuation)) #map punctuation to space
    return text.translate(table)

In [ ]:
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

In [ ]:
def remove_non_ascii(text):
    """Remove non-ASCII characters from list of tokenized words"""
    new_text = []
    text = text.split()
    for word in text:
        new_word = unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8', 'ignore')
        new_text.append(new_word)
    new_text = ' '.join(new_text)
    return new_text

In [ ]:
exception_words = ['few', 'more', 'most', 'because', 'now', 'before', 'after',
                   'what', 'when', 'where', 'how', 'should', 'which', 'whose']
stop_words = [w for w in stopwords.words('english') if not w in exception_words]

def remove_stopwords(text):
    word_tokens = word_tokenize(text)

    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    filtered_sentence = ' '.join(filtered_sentence)
    return filtered_sentence 

In [ ]:
def clean_text(text):
    text = remove_punctuation(text.lower())
    text = strip_html(text)
    text = remove_non_ascii(text)
    text = remove_stopwords(text)
    result = text.replace(r"\b\w\b","") # remove single char
    return result

In [ ]:
## Cleansing text
data['clean_question1'] = data.question1.progress_apply(clean_text)
data['clean_question2'] = data.question2.progress_apply(clean_text)

In [ ]:
data.head()

In [ ]:
data.to_csv('drive/My Drive/Colab Notebooks/quora-question-pairs/data/clean_quora_duplicate_questions.csv', index=False)